# Trying the model on the interop photos

Each photo is framed for the network by `eyemodel3d.interop`, then predicted.
- `MODE = "letterbox"`: the whole photo, as is (for checkpoints trained with zooms).
- `MODE = "iris"`: reframed so that the iris sits on the dashed circle, from per-dataset guesses
  that you can correct per photo in the viewer (for checkpoints trained without zooms).
The default follows the checkpoint. Old checkpoints that do not predict laterality get left eyes
mirrored; otherwise the laterality from the file name (OD / OS / unknown) is compared with the
network's prediction.

1. gallery per dataset: original photo with the predicted outlines | network input | prediction
2. laterality accuracy per dataset
3. interactive viewer (iris framing, laterality, save to the overrides CSV)
4. predicted parameters of the current photo

Outlines: white = sclera, cyan = iris, yellow = pupil, magenta = caruncle. Grey in the prediction = padding.
The pupil is shown for checkpoints that predict it (trained with `--pupil`); for the others it is a fixed
size and merged with the iris.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image

from eyemodel3d import train as T
import eyemodel3d.renderer.eye_model as em

warnings.filterwarnings("ignore", "Mean of empty slice")  # eyes with no foreground at all
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from eyemodel3d import interop as I
from IPython.display import display

# ---- edit these
CKPT = ROOT / "runs/vit_s_pupil/best.pt"
DATA = I.DEFAULT_ROOT
OVERRIDES = ROOT / "interop_overrides.csv"  # per-photo fixes for the "iris" framing
MODE = None  # "letterbox" / "iris"; None = letterbox if the checkpoint was trained with zooms

In [ ]:
net, space, fixed, meta = T.load_model(CKPT, dev)
SIZE = tuple(meta["size"])
H, W = SIZE
PREDICTS_LAT = "medial" not in fixed
MIRROR = not PREDICTS_LAT
PREDICTS_PUPIL = "r_pupil" not in fixed
MODE = MODE or ("letterbox" if meta.get("zoom", {}).get("p_zoom", 0) > 0 else "iris")

overrides = I.load_overrides(OVERRIDES) if OVERRIDES.exists() else {}
records = I.list_images(DATA, overrides=overrides)
by_key = {r.key: r for r in records}
datasets = sorted({r.dataset for r in records})
print(f"checkpoint epoch {meta['epoch']} (val Dice {meta['val_dice']:.3f}), input {H}x{W}, "
      f"framing {MODE!r}, laterality {'predicted' if PREDICTS_LAT else 'fixed: left eyes mirrored'}, "
      f"pupil {'predicted' if PREDICTS_PUPIL else 'not predicted'}")
print(f"{len(records)} photos in {datasets}; {len(overrides)} with overrides")

In [ ]:
PUPIL = T.NT  # extra label value for the pupil in the label maps
OUTLINE = {1: "w", 2: "c", 3: "m", PUPIL: "y"}  # label -> outline colour


def colorize(lbl):
    """T.colorize with the pupil in the renderer's pupil colour."""
    rgb = T.colorize(np.where(lbl == PUPIL, T.IRIS, lbl))
    rgb[lbl == PUPIL] = em.COLORS[em.CLASSES.index("pupil")].numpy()
    return rgb


@torch.no_grad()
def run(record):
    """Frame, predict. Returns prep, label map (H, W) (pupil = PUPIL if the checkpoint
    predicts it), parameter values, laterality, P(OD)."""
    prep = I.prepare(record, SIZE, MODE, MIRROR)
    z, lat = net(I.to_tensor(prep)[None].to(dev))
    v = space.values(z)
    m = T.predicted_medial(lat, fixed)
    probs, pp = T.render_batch(space.as_dict(v), fixed, m, H, W, meta["tau_min"], pupil=True)
    lbl = probs[0].argmax(-1).cpu().numpy().astype(np.uint8)
    if PREDICTS_PUPIL:
        lbl[(pp[0] > 0.5).cpu().numpy() & (lbl == T.IRIS)] = PUPIL
    lat_pred = "OD" if m[0] > 0 else "OS"
    if prep.mirrored:  # the network saw a mirrored eye
        lat_pred = {"OD": "OS", "OS": "OD"}[lat_pred]
    p_od = float(torch.sigmoid(lat[0])) if PREDICTS_LAT else float("nan")
    return prep, lbl, v[0].cpu().numpy(), lat_pred, p_od


def outlines(ax, lbl, x=None, y=None):
    for c, col in OUTLINE.items():
        m = lbl == c
        if m.any():
            if x is None:
                ax.contour(m, levels=[0.5], colors=col, linewidths=1)
            else:
                ax.contour(x, y, m, levels=[0.5], colors=col, linewidths=1.2)


def draw(record, axs):
    prep, lbl, _, lat_pred, p_od = run(record)
    # original photo, prediction mapped back (subsampled for speed)
    W0, H0 = prep.source_size
    k = max(1, max(W0, H0) // 700)
    back = I.to_source(lbl, prep)[::k, ::k]
    src = Image.open(record.path).convert("RGB").resize((back.shape[1], back.shape[0]))
    axs[0].imshow(src, extent=(0, W0, H0, 0))
    outlines(axs[0], back, np.arange(back.shape[1]) * k, np.arange(back.shape[0]) * k)
    wrong = record.laterality != "unknown" and record.laterality != lat_pred
    axs[0].set_title(f"{record.key}   file: {record.laterality}   predicted: {lat_pred}"
                     + (f" (P(OD) {p_od:.2f})" if PREDICTS_LAT else ""),
                     fontsize=8, color="r" if wrong else "k")
    # network input with outlines (and the target iris circle in the iris framing)
    axs[1].imshow(prep.image)
    outlines(axs[1], lbl)
    if MODE == "iris":
        axs[1].add_patch(plt.Circle((I.TARGET_CENTER[0] * W, I.TARGET_CENTER[1] * H),
                                    I.TARGET_IRIS_W * W / 2, fill=False, ls="--", color="y"))
    axs[1].set_title("network input" + (" (mirrored)" if prep.mirrored else ""), fontsize=8)
    shown = lbl.copy()
    shown[~prep.valid] = T.IGNORE
    axs[2].imshow(colorize(shown))
    axs[2].set_title("prediction", fontsize=8)
    for a in axs:
        a.set_axis_off()


def gallery(dataset, n=6, start=0):
    recs = [r for r in records if r.dataset == dataset][start:start + n]
    fig, axs = plt.subplots(len(recs), 3, figsize=(13, 3.2 * len(recs)), squeeze=False,
                            gridspec_kw=dict(width_ratios=[1.3, 1, 1]))
    for r, row in zip(recs, axs):
        draw(r, row)
    plt.tight_layout()
    plt.show()

## 1. Gallery\nChange `DATASET` / `START` and re-run. A red title = predicted laterality differs from the file name.

In [ ]:
print(datasets)
DATASET = "sbvpi"
START = 0
gallery(DATASET, n=6, start=START)

## 2. Laterality per dataset
Against the laterality the photo looks like according to its file name (sbvpi names are mirrored; see interop/core.py). Datasets without it only show the predicted split.

In [ ]:
if not PREDICTS_LAT:
    print("This checkpoint does not predict laterality (left eyes are mirrored from the file names).")
else:
    loader = torch.utils.data.DataLoader(I.make_dataset(records, SIZE, MODE, MIRROR),
                                         batch_size=16, num_workers=4)
    p_od = []
    with torch.no_grad():
        for x, _, _ in loader:
            p_od += torch.sigmoid(net(x.to(dev))[1]).tolist()
    lat_df = pd.DataFrame(dict(dataset=[r.dataset for r in records], name=[r.path.name for r in records],
                               file=[r.laterality for r in records], p_od=p_od))
    lat_df["predicted"] = np.where(lat_df.p_od >= 0.5, "OD", "OS")
    lat_df["correct"] = np.where(lat_df.file == "unknown", np.nan, lat_df.file == lat_df.predicted)
    known = lat_df[lat_df.file != "unknown"]
    print(f"accuracy on photos with known laterality: {known.correct.mean():.3f} ({len(known)} photos)")
    display(lat_df.groupby("dataset").agg(n=("name", "size"),
                                          predicted_OD=("predicted", lambda s: (s == "OD").mean()),
                                          accuracy=("correct", "mean")).round(3))

## 2b. Pupil per dataset
Predicted pupil size (pupil radius / limbus radius, as a fraction of the iris) and pupil offset from the
iris centre (mm). In the training data (detected pupils) the ratio is about 0.28-0.32.

In [ ]:
if not PREDICTS_PUPIL:
    print("This checkpoint does not predict the pupil (train with --pupil).")
else:
    loader = torch.utils.data.DataLoader(I.make_dataset(records, SIZE, MODE, MIRROR),
                                         batch_size=16, num_workers=4)
    vals = []
    with torch.no_grad():
        for x, _, _ in loader:
            vals.append(space.values(net(x.to(dev))[0]).cpu())
    vals = torch.cat(vals)
    pv = {n: vals[:, i].numpy() for i, n in enumerate(space.names)}
    r_lim = pv["r_limbus"] if "r_limbus" in pv else np.full(len(vals), meta["fixed"]["r_limbus"])
    pup_df = pd.DataFrame(dict(dataset=[r.dataset for r in records], name=[r.path.name for r in records],
                               ratio=pv["r_pupil"] / r_lim, r_pupil=pv["r_pupil"],
                               dx=pv.get("dx", 0.0), dy=pv.get("dy", 0.0)))
    display(pup_df.groupby("dataset")[["ratio", "r_pupil", "dx", "dy"]]
            .describe(percentiles=[0.5]).loc[:, (slice(None), ["mean", "50%", "std"])].round(2))
    pup_df.boxplot("ratio", by="dataset", figsize=(9, 3), rot=30)
    plt.suptitle("")
    plt.title("pupil / iris radius")
    plt.show()

## 3. Interactive viewer
Pick a photo and check the prediction. In the `"iris"` framing, move the sliders until the iris sits
on the dashed circle; the laterality buttons set the reference (and the mirroring for old
checkpoints). **Save** writes the photo's values to the overrides CSV. Sliders are fractions of the
original photo (iris width = iris diameter / photo width).

In [ ]:
import ipywidgets as wd
from IPython.display import display

edited = set(overrides)
names = [r.key for r in records]
w_img = wd.Dropdown(options=names, description="photo", layout=wd.Layout(width="480px"))
w_cx = wd.FloatSlider(min=0, max=1, step=0.005, description="iris x", continuous_update=False)
w_cy = wd.FloatSlider(min=0, max=1, step=0.005, description="iris y", continuous_update=False)
w_w = wd.FloatSlider(min=0.05, max=1.5, step=0.005, description="iris width", continuous_update=False)
w_lat = wd.ToggleButtons(options=["OD", "OS", "unknown"], description="laterality")
w_prev = wd.Button(description="◀ prev")
w_next = wd.Button(description="next ▶")
w_save = wd.Button(description="Save to overrides", button_style="success")
w_reset = wd.Button(description="Reset to default")
w_msg = wd.Label()
out = wd.Output()
state = {"loading": False}
for w in (w_cx, w_cy, w_w):
    w.disabled = MODE != "iris"


def current():
    return by_key[w_img.value]


def load_sliders(*_):
    r = current()
    state["loading"] = True
    w_cx.value, w_cy.value, w_w.value, w_lat.value = r.iris_cx, r.iris_cy, r.iris_w, r.laterality
    state["loading"] = False
    w_msg.value = "has overrides" if r.key in edited else "dataset defaults"
    refresh()


def refresh(*_):
    if state["loading"]:
        return
    r = current()
    r.iris_cx, r.iris_cy, r.iris_w, r.laterality = w_cx.value, w_cy.value, w_w.value, w_lat.value
    with out:
        out.clear_output(wait=True)
        fig, axs = plt.subplots(1, 3, figsize=(14, 4), gridspec_kw=dict(width_ratios=[1.3, 1, 1]))
        draw(r, axs)
        plt.tight_layout()
        plt.show()


def save(_):
    edited.add(current().key)
    I.write_template([by_key[k] for k in names if k in edited], OVERRIDES)
    w_msg.value = f"saved ({len(edited)} photos in {OVERRIDES.name})"


def reset(_):
    r = current()
    info = I.DATASETS.get(r.dataset, I.core.GENERIC)
    r.iris_cx, r.iris_cy = info.center
    r.iris_w, r.laterality = info.iris_w, info.laterality(r.path.name)
    if r.key in edited:
        edited.discard(r.key)
        I.write_template([by_key[k] for k in names if k in edited], OVERRIDES)
    load_sliders()


def step(d):
    def go(_):
        w_img.index = (w_img.index + d) % len(names)
    return go


w_img.observe(load_sliders, "value")
for w in (w_cx, w_cy, w_w, w_lat):
    w.observe(refresh, "value")
w_save.on_click(save)
w_reset.on_click(reset)
w_prev.on_click(step(-1))
w_next.on_click(step(1))

display(wd.VBox([
    wd.HBox([w_img, w_prev, w_next]),
    wd.HBox([w_cx, w_cy, w_w]),
    wd.HBox([w_lat, w_save, w_reset, w_msg]),
    out,
]))
load_sliders()

## 4. Predicted parameters of the photo shown above

In [ ]:
def param_table(vals):
    """Predicted values vs the checkpoint's bounds; the bar shows the position in the range
    (log scale for f). A value pinned at a bound often means the model lacks something."""
    v = torch.as_tensor(vals, device=dev)[None]
    pos = space.normalize(v)[0].cpu().numpy()
    rows = []
    for j, n in enumerate(space.names):
        k = round(20 * float(np.clip(pos[j], 0, 1)))
        rows.append(dict(param=n, value=round(float(vals[j]), 3),
                         default=round(float(space.default[j]), 3),
                         lo=round(float(space.lo[j]), 3), hi=round(float(space.hi[j]), 3),
                         position="|" + "#" * k + "." * (20 - k) + "|"))
    for n, x in meta["fixed"].items():
        rows.append(dict(param=n, value=round(x, 3), default="fixed"))
    return pd.DataFrame(rows).set_index("param")


_, _, vals, lat_pred, _ = run(current())
print(current().key, "predicted", lat_pred)
param_table(vals)